<a href="https://colab.research.google.com/github/hosseinta2/RL/blob/main/RL_cross_entropy_method.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# Cross entropy method

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset
import random
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from IPython.display import Video
import glob
import numpy as np

class RandomActionWrapper(gym.ActionWrapper):
  def __init__(self, env: gym.Env, epsilon: float= 0.05):
    super(RandomActionWrapper,self).__init__(env)
    self.epsilon = epsilon
  def action(self, action: gym.core.WrapperActType):
    if random.random()<self.epsilon:
       action = self.env.action_space.sample()
    return action

class NN(nn.Module):
  def __init__(self, input_size, hidden_size, output_size) -> None:
    super().__init__()
    self.net = nn.Sequential(nn.Linear(input_size,hidden_size),nn.ReLU(),nn.Linear(hidden_size,output_size))
  def forward(self,x):
    return self.net(x)

def batches(env:gym.Env, net, batch_size,device):
  with torch.no_grad():

      reward_v = []
      episode_pairs = []
      for _ in range(batch_size):
        reward_total = 0
        obs,_= env.reset()
        x = torch.tensor([]).to(device)
        y = torch.tensor([],dtype = torch.long).to(device)
        while True:
          obs = torch.tensor(obs, dtype = torch.float32).to(device)
          act_scores = net(obs.unsqueeze(0))
          #act_probs = torch.softmax(act_scores, dim=-1)
          dist = torch.distributions.Categorical(logits=act_scores)
          action = dist.sample()
          next_obs,reward,is_terminated,is_truncated,_ = env.step(action.item())
          reward_total += reward
          x = torch.cat((x,obs.unsqueeze(0)),dim=0)
          y = torch.cat((y,action.unsqueeze(0)),dim=0)
          obs = next_obs
          if is_terminated or is_truncated:
            episode_pairs.append((x,y))
            reward_v.append(reward_total)
            break
  return (episode_pairs,reward_v)

def elite_episodes(pairs,rewards,percentile,device):
    mean_reward = sum(rewards)/len(rewards)
    threshold = np.percentile(rewards, percentile)

    elite_idx = [i for i, r in enumerate(rewards) if r >= threshold]

    features = torch.tensor([]).to(device)
    labels = torch.tensor([],dtype=torch.long).to(device)
    for i in elite_idx:
      x,y = pairs[i]
      features = torch.cat((features,x),dim=0)
      labels = torch.cat((labels,y),dim=0)
    return features,labels.squeeze(),mean_reward
hidden_size = 100
batch_size = 20
env = RandomActionWrapper(gym.make("CartPole-v1", render_mode = "rgb_array"))
#env = RecordVideo(env,video_folder="videos",episode_trigger=lambda episode_id: True)
neural_net = NN(input_size=4,hidden_size=hidden_size,output_size=2)
obj = nn.CrossEntropyLoss()
optimizer = optim.Adam(params = neural_net.parameters(),lr= 0.01)
total_steps = 60
percentile = 80
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
neural_net.to(device)
reward_vec = []
loss_vec = []
for t in range(total_steps):
    pairs,rewards = batches(env, neural_net, batch_size,device)
    features,labels,mean_reward = elite_episodes(pairs,rewards,percentile,device)
    features = features.to(device)
    labels = labels.to(device)
    optimizer.zero_grad()
    output = neural_net(features)
    loss_val= obj(output,labels)
    loss_val.backward()
    optimizer.step()
    loss_vec.append(loss_val.item())
    reward_vec.append(mean_reward)
    if t%10==0:
       print("step = ", t,"  loss = ", loss_vec[-1], "  mean reward = ", reward_vec[-1])




cuda
step =  0   loss =  0.6921225786209106   mean reward =  22.15
step =  10   loss =  0.6091016530990601   mean reward =  49.85
step =  20   loss =  0.5659810304641724   mean reward =  59.35
step =  30   loss =  0.5430845022201538   mean reward =  150.75
step =  40   loss =  0.5392470955848694   mean reward =  194.45
step =  50   loss =  0.5238320231437683   mean reward =  304.85


In [29]:
# Testing the model

import os
import glob
from IPython.display import Video, display
from gymnasium.wrappers import RecordVideo

#remove old videos
for f in glob.glob("videos/*.mp4"):
    os.remove(f)

test_env = gym.make("CartPole-v1", render_mode="rgb_array")
test_env = RecordVideo(
    test_env,
    video_folder="videos",
    episode_trigger=lambda episode_id: True
)

obs, _ = test_env.reset()
total_reward = 0

neural_net.eval()

while True:
    obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

    with torch.no_grad():
        logits = neural_net(obs_t)

        # For testing, use greedy action instead of random sampling
        action = torch.argmax(logits, dim=1).item()

    obs, reward, terminated, truncated, _ = test_env.step(action)

    total_reward += reward

    if terminated or truncated:
        break

test_env.close()

print("test reward:", total_reward)

video_path = glob.glob("videos/*.mp4")[-1]
display(Video(video_path, embed=True))

test reward: 500.0
